In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from expand_subgraph_freebase import ExpandSubgraphFreebase
from model_2 import GNN_auto, Projector
import pickle as pkl
import numpy as np
import torch
from load_data import DataLoader
# from loader2 import DataLoader2 as GnnDataLoader
from reasoning_with_fb import ReasoningModuleFreebase as ReasoningModule
from freebase_interface import freebase_interface
import time
import random

In [ ]:
# test_query = {
#     "ID": "WebQTest-832_c334509bb5e02cacae1ba2e80c176499",
#     "compositionality_type": "composition",
#     "created": "2018-02-13T04:12:57",
#     "machine_question": "when is the last time the the team has a team moscot named Lou Seal won the world series",
#     "question": "Lou Seal is the mascot for the team that last won the World Series when?",
#     "sparql": "PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT DISTINCT ?x\nWHERE {\nFILTER (?x != ?c)\nFILTER (!isLiteral(?x) OR lang(?x) = '' OR langMatches(lang(?x), 'en'))\n?c ns:sports.sports_team.team_mascot ns:m.03_dwn . \n?c ns:sports.sports_team.championships ?x .\n?x ns:time.event.start_date ?sk0 .\n}\nORDER BY DESC(xsd:datetime(?sk0))\nLIMIT 1\n",
#     "webqsp_ID": "WebQTest-832",
#     "webqsp_question": "when is the last time the giants won the world series",
#     "topic_entity": {
#         "m.03_dwn": "Lou Seal"
#     },
#     "answer": [
#         "2014 World Series"
#     ],
#     "qid_topic_entity": {},
#     "mid_crucial_triples": [],
#     "idx_in_processed_samples": 0,
#     "index": 0,
#     "crucial_triples": []
# }

# queries = [test_query]

In [ ]:
import json
queries = json.load(open("knowledge_graph/queries/data_with_ct_0.6.json", "r"))

#### Examples:
Example 1: 153  
Example 2: 78  
Example 3: 159
Example 4: 161

In [6]:
class Config:
    data_path = 'knowledge_graph/KG_data/FB15k-237-betae'
    seed = 1234
    k_rel = 4 # beams
    k_cands = 120
    depth = 2 # max depth of subgraph
    cands_lim = 1024
    gpu = 0
    fact_ratio = 1.0
    val_num = -1 # how many triples are used as the validate set
    add_manual_edges = False
    remove_1hop_edges = True
    not_shuffle_train = False
    device = "cuda:0"

In [ ]:
args = Config()

544230 0
==> removing 1-hop links...
==> done


In [ ]:
GoG_args = {
    'drop_ratio': 0.4,
}

sampler = ExpandSubgraphFreebase(
    args,
    GoG_simulation=True,
    GoG_args=GoG_args
)

In [ ]:
def check_reachability(id_query, subgraph_data):
    summ = 0
    for ans in queries[id_query]['answers_id']:
        cnt = 0
        for edges in subgraph_data[2]:
            if ans in edges[[0,2]]:
                cnt += 1

        summ += cnt
        return summ

In [13]:
class GNN_config:
    n_layer = 3
    hidden_dim = 64
    attn_dim = 4
    dropout = 0.3
    n_ent = 1024 # subgraph size
    n_rel = 237
    train_ver = 20
    shortcut = True
    readout = 'linear'
    concatHidden = True
    initializer = 'relation'
    
class Projector_config:
    n_layers = 4
    in_dim = 4096
    hidden_dims = [512, 256]
    out_dim = 64

In [ ]:
modelPath = "weights/finetune/topk_ValMRR_0.269.pt"
checkpoint = torch.load(modelPath, map_location=torch.device("cuda:0"), weights_only=False)


gnn_model = GNN_auto(GNN_config)
gnn_model.load_state_dict(checkpoint['gnn_model'])
gnn_model = gnn_model.to(torch.device("cuda:0"))

In [15]:


projector_model = Projector(Projector_config.in_dim, Projector_config.hidden_dims, Projector_config.out_dim, Projector_config.n_layers)
projector_model.load_state_dict(checkpoint['projector'])
projector_model = projector_model.to(torch.device("cuda:0"))

In [ ]:
reasoning_module = ReasoningModule(sampler,
                                   gnn_model,
                                   projector_model,
                                   "gpt-3.5-turbo") 
# gpt-4o, gpt-3.5-turbo, "meta-llama/Llama-3.1-8B-Instruct"


## Run

In [ ]:
subgraph_reachability = []
results = []
ground_truths = []
for idx, query in enumerate(queries[:1]):
    reasoning_module.assign_query(query)
    subgraph_data = reasoning_module.subgraph_data
    # ent_dict, rel_dict, rel_desc_dict = create_dicts(subgraph_data)
    is_reachable = check_reachability(idx, subgraph_data)
    subgraph_reachability.append(is_reachable)
    if is_reachable == 0:
        results.append(None)
        ground_truths.append(query['answers_id'])
        continue
    # reasoning_module.assign_dict(ent_dict, rel_dict, rel_desc_dict)
    reasoning_module.reasoning()
    results.append(reasoning_module.result)
    ground_truths.append(query['answers_id'])
    time.sleep(1)

Simulating GoG by removing edges...
Simulating GoG by removing edges...
Hop 1 with 1 active path(s)
path: 0 {'current_entity': 6290, 'trace': []}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
4 new paths generated at hop 1
Hop 1: 4 active path(s)
Hop 2 with 4 active path(s)
path: 0 {'current_entity': 10176, 'trace': [(6290, 'Identifies the films in which the actor has performed.', 10176)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
path: 1 {'current_entity': 2125, 'trace': [(6290, 'Identifies the films in which the actor has performed.', 2125)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
path: 2 {'current_entity': 1431, 'trace': [(6290, 'Identifies the TV programs in which the actor had a regular cast role', 1431)]}
------ call_llm_judge:
Answer found: {YES}. Answer: ["The Office"]
